In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

In [4]:
RESULTS_DIR = "../evaluation/results"
IMAGES_DIR = "./images"
Path(IMAGES_DIR).mkdir(parents=True, exist_ok=True)

os.listdir(RESULTS_DIR)

['gemma3_latest_async_chain_results_validation.csv',
 'gemma3_latest_sync_chain_results_validation.csv',
 'llama3_2_async_chain_results_validation.csv',
 'llama3_2_sync_chain_results_validation.csv']

In [5]:
def load_benchmark_df(results_dir: str) -> pd.DataFrame:
    benchmark_df = pd.DataFrame()
    for filename in os.listdir(results_dir):
        df = pd.read_csv(os.path.join(results_dir, filename))
        model_name = "_".join(filename.split("/")[-1].split("_")[0:2])
        df["ModelName"] = model_name
        is_async = "async" in filename
        df["Algorithm"] = "Async" if is_async else "Sync"
        benchmark_df = pd.concat([benchmark_df, df])
    return benchmark_df

benchmark_df = load_benchmark_df(RESULTS_DIR)
benchmark_df.head()

,RowID,ErrorOccurred,ExecutionOutput,HasTimedOut,ExecutionError,ExecutionTime,ExpectedOutput,ActualOutput,CorrectOutput,FirstOutput,ModelName,Algorithm
0,511,False,192\n,False,NaN,2.369337,15,192,False,2.258914,gemma3_latest,Async
1,511,False,7\n,False,NaN,2.353086,2,7,False,1.679477,gemma3_latest,Async
2,512,True,NaN,False,Syntax error detected. Halting further process...,1.642581,NaN,NaN,False,1.642581,gemma3_latest,Async
3,512,True,NaN,False,Syntax error detected. Halting further process...,1.929561,NaN,NaN,False,1.929561,gemma3_latest,Async
4,513,False,"[7, 'PF', 8, 'PF', 9, 'PF', 10, 'PF']\n",False,NaN,2.362301,"[7, 'PF', 8, 'PF', 9, 'PF', 10, 'PF']","[7, 'PF', 8, 'PF', 9, 'PF', 10, 'PF']",True,2.216615,gemma3_latest,Async


In [6]:
def display_execution_time_describe(df: pd.DataFrame) -> None:
    print(df.groupby(["ModelName", "Algorithm"])["ExecutionTime"].describe())

display_execution_time_describe(benchmark_df)

                         count      mean       std       min       25%  \
ModelName     Algorithm                                                  
gemma3_latest Async       10.0  2.027377  0.261422  1.642581  1.898498   
              Sync        10.0  2.288260  0.275337  1.855144  2.088186   
llama3_2      Async       10.0  1.692440  0.124921  1.496345  1.606867   
              Sync        10.0  1.710941  0.130674  1.540312  1.596705   

                              50%       75%       max  
ModelName     Algorithm                                
gemma3_latest Async      1.945326  2.291026  2.369337  
              Sync       2.253877  2.476485  2.663895  
llama3_2      Async      1.671800  1.806184  1.875529  
              Sync       1.687823  1.818681  1.891498  


In [7]:
def save_summary_tables(df: pd.DataFrame, tables_dir: str) -> None:
    Path(tables_dir).mkdir(parents=True, exist_ok=True)
    grouped = df.groupby(["ModelName", "Algorithm", "ErrorOccurred"]).agg(
        Count=("ExecutionTime", "count"),
        Avg_ExecTime=("ExecutionTime", "mean"),
        Median_ExecTime=("ExecutionTime", "median"),
        Avg_FirstOutput=("FirstOutput", "mean"),
        Median_FirstOutput=("FirstOutput", "median")
    ).reset_index()
    grouped.to_csv(os.path.join(tables_dir, "summary_by_error.csv"), index=False)

save_summary_tables(benchmark_df, "./tables")

def display_summary_tables(df: pd.DataFrame) -> None:
    grouped = df.groupby(["ModelName", "Algorithm", "ErrorOccurred"]).agg(
        Count=("ExecutionTime", "count"),
        Avg_ExecTime=("ExecutionTime", "mean"),
        Median_ExecTime=("ExecutionTime", "median"),
        Avg_FirstOutput=("FirstOutput", "mean"),
        Median_FirstOutput=("FirstOutput", "median")
    ).reset_index()
    non_error_df = df[df["ErrorOccurred"] == False]
    grouped_correct = non_error_df.groupby(["ModelName", "Algorithm", "CorrectOutput"]).agg(
        Count=("ExecutionTime", "count"),
        Avg_ExecTime=("ExecutionTime", "mean"),
        Median_ExecTime=("ExecutionTime", "median"),
        Avg_FirstOutput=("FirstOutput", "mean"),
        Median_FirstOutput=("FirstOutput", "median")
    ).reset_index()
    error_summary = df.groupby(["ModelName", "Algorithm"]).agg(
        ErrorCount=("ErrorOccurred", lambda x: (x == True).sum())
    ).reset_index()
    final = pd.merge(grouped, error_summary, on=["ModelName", "Algorithm"], how="left")
    print("Summary by ModelName, Algorithm, and ErrorOccurred:")
    display(final)
    print("\nSummary by ModelName, Algorithm, and CorrectOutput (non-error cases only):")
    display(grouped_correct)

display_summary_tables(benchmark_df)

Summary by ModelName, Algorithm, and ErrorOccurred:


,ModelName,Algorithm,ErrorOccurred,Count,Avg_ExecTime,Median_ExecTime,Avg_FirstOutput,Median_FirstOutput,ErrorCount
0,gemma3_latest,Async,False,6,2.166382,2.228966,1.940433,1.867504,4
1,gemma3_latest,Async,True,4,1.818868,1.835901,1.818868,1.835901,4
2,gemma3_latest,Sync,False,10,2.288260,2.253877,2.288140,2.253767,0
3,llama3_2,Async,False,10,1.692440,1.671800,1.509600,1.499698,0
4,llama3_2,Sync,False,10,1.710941,1.687823,1.710847,1.687736,0



Summary by ModelName, Algorithm, and CorrectOutput (non-error cases only):


,ModelName,Algorithm,CorrectOutput,Count,Avg_ExecTime,Median_ExecTime,Avg_FirstOutput,Median_FirstOutput
0,gemma3_latest,Async,False,3,2.275756,2.353086,1.965927,1.959390
1,gemma3_latest,Async,True,3,2.057008,1.916087,1.914938,1.775619
2,gemma3_latest,Sync,False,5,2.300982,2.379471,2.300864,2.379339
3,gemma3_latest,Sync,True,5,2.275538,2.128283,2.275416,2.128194
4,llama3_2,Async,False,5,1.675139,1.654009,1.512616,1.515835
5,llama3_2,Async,True,5,1.709742,1.779952,1.506585,1.497984
6,llama3_2,Sync,False,5,1.682771,1.658647,1.682680,1.658562
7,llama3_2,Sync,True,5,1.739111,1.801741,1.739013,1.801644


In [8]:
def plot_execution_time_catplot(df: pd.DataFrame, image_path: str) -> None:
    sns.set_style(style="whitegrid")
    g_exec = sns.catplot(
        data=df, 
        x="Algorithm", 
        y="ExecutionTime", 
        hue="ErrorOccurred", 
        col="ModelName", 
        kind="box",
        height=4, 
        aspect=0.8,
        palette="Set2",
        sharey=False
    )
    g_exec.set_titles("Model: {col_name}")
    g_exec.figure.suptitle("Execution Time by Algorithm and Error Occurrence", y=1.05)
    g_exec.set_axis_labels("Algorithm", "Execution Time (s)")
    plt.tight_layout()
    g_exec.savefig(image_path)
    plt.close()

def plot_first_output_catplot(df: pd.DataFrame, image_path: str) -> None:
    sns.set_style(style="whitegrid")
    g_first = sns.catplot(
        data=df, 
        x="Algorithm", 
        y="FirstOutput", 
        hue="ErrorOccurred", 
        col="ModelName", 
        kind="box",
        height=4, 
        aspect=0.8,
        palette="Set2",
        sharey=False
    )
    g_first.set_titles("Model: {col_name}")
    g_first.figure.suptitle("First Output Time by Algorithm and Error Occurrence", y=1.05)
    g_first.set_axis_labels("Algorithm", "First Output Time (s)")
    plt.tight_layout()
    g_first.savefig(image_path)
    plt.close()

plot_execution_time_catplot(benchmark_df, os.path.join(IMAGES_DIR, "execution_time_catplot.png"))
plot_first_output_catplot(benchmark_df, os.path.join(IMAGES_DIR, "first_output_catplot.png"))

In [9]:
def plot_execution_time_hist(df: pd.DataFrame, image_path: str) -> None:
    plt.figure(figsize=(10, 6))
    async_df = df[df["Algorithm"] == "Async"]
    sync_df = df[df["Algorithm"] == "Sync"]
    sns.histplot(x=async_df["ExecutionTime"], bins=20, kde=True, label="Async")
    sns.histplot(x=sync_df["ExecutionTime"], bins=20, kde=True, label="Sync")
    plt.title("Distribution of Execution Times")
    plt.xlabel("Execution Time (seconds)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(image_path)
    plt.close()

plot_execution_time_hist(benchmark_df, os.path.join(IMAGES_DIR, "execution_time_hist.png"))

In [10]:
def plot_first_output_hist(df: pd.DataFrame, image_path: str) -> None:
    plt.figure(figsize=(10, 6))
    async_df = df[df["Algorithm"] == "Async"]
    sync_df = df[df["Algorithm"] == "Sync"]
    sns.histplot(x=async_df["FirstOutput"], bins=20, kde=True, label="Async")
    sns.histplot(x=sync_df["FirstOutput"], bins=20, kde=True, label="Sync")
    plt.title("Distribution of First Output Times")
    plt.xlabel("First Output Time (seconds)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(image_path)
    plt.close()

plot_first_output_hist(benchmark_df, os.path.join(IMAGES_DIR, "first_output_hist.png"))

In [11]:
def plot_execution_time_boxplot(df: pd.DataFrame, image_path: str) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    df.boxplot(column="ExecutionTime", by=["ErrorOccurred", "Algorithm"], ax=ax)
    plt.title("Execution Time by Error Occurrence and Algorithm")
    plt.suptitle("")
    plt.xlabel("Error Occurred and Algorithm")
    plt.ylabel("Execution Time (seconds)")
    plt.tight_layout()
    plt.savefig(image_path)
    plt.close()

plot_execution_time_boxplot(benchmark_df, os.path.join(IMAGES_DIR, "execution_time_boxplot.png"))